In [ ]:
from qgravnet import QGravNetFactory

from utils.data import load_processed
from utils.evaluation import load_run
from utils.files import HLS4ML_OUT_PATH, PROJECT_ROOT, RESULTS_PATH

In [ ]:
model_cfg, weights_path, history, datapath = load_run(RESULTS_PATH / 'train_new_quantization_cfg_y_train_scaled')

D = load_processed(datapath)

trained_model = QGravNetFactory(**model_cfg).create_keras_model(128, 4)
trained_model.load_weights(weights_path)


test_energy_pred, test_pid_pred = trained_model.predict(D['X_hits_test'])
test_energy_pred *= 100

In [ ]:
import hls4ml
from hls4ml_gravnet.hls4ml_extension.global_exchange import HGlobalExchange
from hls4ml_gravnet.hls4ml_extension.global_exchange_parser import parse_global_exchange
from hls4ml_gravnet.hls4ml_extension.global_exchange_template import (
    GlobalExchangeConfigTemplate,
    GlobalExchangeFunctionTemplate,
)
from hls4ml_gravnet.hls4ml_extension.gravnet_core import HGravNetCore
from hls4ml_gravnet.hls4ml_extension.gravnet_core_parser import parse_gravnet_layer
from hls4ml_gravnet.hls4ml_extension.gravnet_core_template import GravNetCoreConfigTemplate, GravNetCoreFunctionTemplate

try:
    hls4ml.converters.register_keras_v2_layer_handler('GravNetCore', parse_gravnet_layer)
    hls4ml.converters.register_keras_v2_layer_handler('GlobalExchange', parse_global_exchange)
    hls4ml.model.layers.register_layer('GravNetCore', HGravNetCore)
    hls4ml.model.layers.register_layer('GlobalExchange', HGlobalExchange)
    backend = hls4ml.backends.get_backend('Vitis')
    backend.register_template(GravNetCoreConfigTemplate)
    backend.register_template(GravNetCoreFunctionTemplate)
    backend.register_template(GlobalExchangeConfigTemplate)
    backend.register_template(GlobalExchangeFunctionTemplate)
    backend.register_source(PROJECT_ROOT / 'hls4ml_gravnet' / 'hls' / 'nnet_gravnet_core.h')
    backend.register_source(PROJECT_ROOT / 'hls4ml_gravnet' / 'hls' / 'nnet_global_exchange.h')
except Exception:
    pass  # Already registered

In [ ]:
from utils.config import set_qgravnet_hls_config

hls_config = hls4ml.utils.config_from_keras_model(
    trained_model,
    granularity='name',
    backend='Vitis',
)
set_qgravnet_hls_config(hls_config)

hls_model = hls4ml.converters.convert_from_keras_model(
    trained_model, hls_config=hls_config, output_dir=str(HLS4ML_OUT_PATH / 'toy_calo'), backend='Vitis'
)
hls_model.compile()

In [ ]:
hls_test_energy_pred, hls_test_pid_pred = hls_model.predict(D['X_hits_test'])
hls_test_energy_pred *= 100

In [ ]:
from utils.evaluation import display_evaluation_results

display_evaluation_results(test_energy_pred=test_energy_pred, test_pid_pred=test_pid_pred, D=D, model_cfg=model_cfg)
display_evaluation_results(test_energy_pred=hls_test_energy_pred, test_pid_pred=hls_test_pid_pred, D=D, model_cfg=model_cfg)

In [ ]:
from hls4ml.model.profiling import get_ymodel_keras

hls_pred, hls_trace = hls_model.trace(D['X_hits_test'])
keras_trace = get_ymodel_keras(trained_model, D['X_hits_test'])

## Tracing & Profiling

Compare min / max outputs of layers for a quick overview

In [ ]:
import numpy as np

for key, value in keras_trace.items():
    print(f'{key}: {np.max(value)}, {np.min(value)}')

print()
for key, value in hls_trace.items():
    print(f'{key}: {np.max(value)}, {np.min(value)}')

Check which layers have the most mismatches (Keras vs HLS)

In [ ]:
ATOL = 4.0

for key in hls_trace.keys():
    if key in keras_trace:
        k_val = keras_trace[key]
        h_val = hls_trace[key]

        diff = np.abs(h_val - k_val)

        if not np.allclose(k_val, h_val, rtol=0, atol=ATOL):
            bad_indices = np.where(diff > ATOL)

            print(f'\n--- MISMATCH FOUND IN LAYER: {key} ---')
            print(f'Max absolute error: {np.max(diff)}')
            print(f'Total mismatch count: {len(bad_indices[0])}')

            print('First 10 mismatches:')
            for i in range(min(len(bad_indices[0]), 10)):
                idx = tuple(d[i] for d in bad_indices)

                k_elem = k_val[idx]
                h_elem = h_val[idx]
                d_elem = diff[idx]

                print(f'  Index {idx}: Keras={k_elem}, HLS={h_elem}, Diff={d_elem}')

            raise AssertionError(f'Abs error for {key} exceeds limit!')

Weight distributions

In [ ]:
from hls4ml.model.profiling import numerical

numerical(model=trained_model, hls_model=hls_model)

Compare prediction distributions

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(test_energy_pred, bins=30, alpha=0.6, label='Keras', color='skyblue', edgecolor='black')

plt.hist(hls_test_energy_pred, bins=30, alpha=0.6, label='HLS', color='salmon', edgecolor='black')

plt.hist(D['y_energy_test'], bins=30, alpha=0.6, label='Ground Truth', color='lightgreen', edgecolor='black')

plt.title('Histogram Comparison of Two Labels')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)

plt.show()